In [12]:
import json
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")
load_dotenv(dotenv_path=env_path)

INPUT_PATH = Path("../../infra/json/graph/current_bellicum.json")
OUTPUT_DIR = Path("../../infra/json/kg_extraction/parse.json")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Input keys:", data.keys())
print("Total nodes:", len(data.get("nodes", [])))

Input keys: dict_keys(['documentId', 'nodes', 'edges'])
Total nodes: 166


In [13]:
nodes = data.get("nodes", [])
edges = data.get("edges", [])

def build_related_paragraphs_json(nodes, edges):
    node_by_id = {
        str(n.get("id", "")): {"text": str(n.get("text", "")).strip()}
        for n in nodes
        if isinstance(n, dict) and n.get("id")
    }

    related_by_id = {node_id: set() for node_id in node_by_id}

    for e in edges:
        if not isinstance(e, dict):
            continue

        if str(e.get("type", "")).strip().lower() != "reference":
            continue

        source_id = str(e.get("source", ""))
        target_id = str(e.get("target", ""))

        if source_id in node_by_id and target_id in node_by_id:
            related_by_id[source_id].add(target_id)
            related_by_id[target_id].add(source_id)  # quita esta línea si lo quieres dirigido

    result = []
    for n in nodes:
        if not isinstance(n, dict) or not n.get("id"):
            continue

        node_id = str(n["id"])
        if node_id not in node_by_id:
            continue

        related_paragraphs = [
            {
                "paragraph_id": rid,
                "text": node_by_id[rid]["text"],
            }
            for rid in related_by_id[node_id]
        ]

        if not related_paragraphs:
            continue

        result.append(
            {
                "paragraph_id": node_id,
                "text": node_by_id[node_id]["text"],
                "related_paragraphs": related_paragraphs,
            }
        )

    return result


result = build_related_paragraphs_json(nodes, edges)

In [14]:
with open(OUTPUT_DIR, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)